# 11. 핫스팟 변이 피처 추가 (H11)

## 배경
- 10 최적 설정(depth 3, lr 0.1, 200그루, colsample 0.3) → CV 0.4113 / LB 0.3206
- 지금 피처(any)는 유전자 단위 0/1 → 같은 유전자의 변이는 종류와 상관없이 모두 1
- 01 EDA 최다 변이: R132H(349), V600E(332), E545K(158), H1047R(154)
  → 특정 변이 자체가 암종을 가리킬 수 있음 (예: IDH1 R132H는 뇌종양, BRAF V600E는 갑상선암·흑색종)

## 가설
- H11: 자주 나오는 '유전자_변이' 조합을 0/1 피처로 추가하면 CV와 LB가 오른다
- H0: 유전자 단위 정보로 충분해서 변이 단위 피처는 효과가 없다

## 검증 방법
- 모델: 10 최적 설정 고정, 피처만 추가 (any + stats + 핫스팟)
- 핫스팟 = fold 학습 부분에서 `min_count`명 이상에게 나온 '유전자_변이' 조합
  - 여러 환자 통계를 쓰므로 Pipeline 안에서 fold마다 새로 고름 (Data Leakage 방지)
  - `min_count` ∈ {5, 10, 20} 비교
- 판단 기준: CV가 10(0.4113)보다 std 이상 오르면 최고 1개 제출, LB가 0.3206보다 오르면 H11 채택
- 데이터: any + stats는 `load_data()`(이소폼 중복 제거 O, test 개수 부풀림 방지), 핫스팟 토큰은 `load_data(dedupe=False)`(표기 통일만, train 핫스팟 후보 보존)

#### 준비: 데이터, 기본 피처, 10 최적 설정
- any + stats는 09·10과 동일
- `BEST`: 10에서 고른 설정 (depth 3, lr 0.1, 200그루, colsample 0.3) → 이번 실험에서는 고정하고 피처만 바꿈

In [1]:
import sys; sys.path.append('..')
import re, time, numpy as np, pandas as pd, xgboost as xgb
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, run_cv, save_submission

seed_everything()
train, test, submission = load_data()              # any + stats용 (dedupe=True)
train_tok, test_tok, _ = load_data(dedupe=False)   # 핫스팟 토큰용 (표기 통일만)
genes = get_gene_cols(train)
y, le = encode_target(train)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

# 10에서 고른 최적 설정 (고정)
BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)

#### 환자별 '유전자_변이' 토큰과 핫스팟 후보 행렬
- `patient_tokens()`: 환자마다 변이를 `BRAF_V600E`처럼 '유전자_변이' 문자열로 바꿔 집합에 담음
  - 한 행 안에서만 계산 → Data Leakage 아님
  - 집합(set)이라 한 칸에 같은 변이가 반복돼도 한 번만 셈
- 후보 줄이기: train 전체에서 5명 이상 나온 조합만 후보로 둠
  - 고유 조합이 약 12만 개라 전부 열로 만들면 메모리 부족
  - fold 학습 부분의 등장 횟수는 전체보다 작거나 같음 → fold에서 5명 이상인 조합은 반드시 후보 안에 있음
  - 따라서 후보를 미리 줄여도 셀 4가 fold 안에서 고르는 결과는 같음 (누수 아님)
- `safe()`: XGBoost 열 이름에 쓸 수 없는 `*`, `>`를 `stop`, `_to_`로 바꿈
- `token_matrix()`: train과 test 모두 같은 후보 목록으로 0/1 행렬을 만듦 → 열 구성 일치

In [2]:
def patient_tokens(df, genes):
    """환자마다 '유전자_변이' 집합. 한 행 안에서만 계산 → 누수 아님.
    set이라 한 칸에 같은 변이가 반복돼도 한 번만 셈."""
    out = []
    for row in df[genes].to_numpy():
        s = set()
        for g, v in zip(genes, row):
            if v != WT:
                s.update(f'{g}_{t}' for t in v.split())
        out.append(s)
    return out

tok_tr, tok_te = patient_tokens(train_tok, genes), patient_tokens(test_tok, genes)

# 후보 목록: train 전체에서 MIN_UNIVERSE명 이상 나온 조합
# fold 학습 부분의 등장 횟수는 전체보다 작거나 같으므로, fold에서 min_count(>=5)를 넘는 조합은
# 반드시 이 후보 안에 있음 → 후보를 미리 줄여도 fold 안 선택 결과는 그대로 (누수 아님)
MIN_UNIVERSE = 5
cnt = Counter(t for s in tok_tr for t in s)
vocab = sorted(t for t, n in cnt.items() if n >= MIN_UNIVERSE)
print(f'고유 조합 {len(cnt):,}개 중 {MIN_UNIVERSE}명 이상: {len(vocab):,}개')
print('상위 10개:', cnt.most_common(10))

def safe(name):  # XGBoost 열 이름에 쓸 수 없는 문자 정리
    return 'hs__' + name.replace('*', 'stop').replace('>', '_to_')

def token_matrix(tok_list, index):
    col = {t: j for j, t in enumerate(vocab)}
    M = np.zeros((len(tok_list), len(vocab)), np.int8)
    for i, s in enumerate(tok_list):
        for t in s:
            if t in col: M[i, col[t]] = 1
    return pd.DataFrame(M, index=index, columns=[safe(t) for t in vocab])

H_tr, H_te = token_matrix(tok_tr, train.index), token_matrix(tok_te, test.index)
X_base,  X_base_test  = pd.concat([F_tr['any'], F_tr['stats']], axis=1), pd.concat([F_te['any'], F_te['stats']], axis=1)
X_hs,    X_hs_test    = pd.concat([X_base, H_tr], axis=1),              pd.concat([X_base_test, H_te], axis=1)
print(X_base.shape, X_hs.shape)

고유 조합 226,795개 중 5명 이상: 429개
상위 10개: [('IDH1_R132H', 345), ('BRAF_V600E', 321), ('PIK3CA_E545K', 149), ('PIK3CA_H1047R', 147), ('TP53_R273C', 92), ('PIK3CA_E542K', 75), ('TP53_R175H', 67), ('TP53_R273H', 61), ('TP53_R248Q', 54), ('TP53_R213*', 47)]
(6201, 4393) (6201, 4822)


#### fold 안에서 핫스팟을 고르는 변환기
- "min_count명 이상에게 나온 변이"를 고르는 건 여러 환자의 통계를 쓰는 것
  → 검증 fold 환자까지 포함해서 고르면 Data Leakage
- `FrequentTokens`: `fit`에서 fold 학습 부분의 등장 횟수를 세고, 기준 미만인 열을 버릴 목록으로 기억 → `transform`에서 그 열을 제거
- `make_hs()`: 변환기와 XGBoost를 Pipeline으로 묶음 → `run_cv`가 fold마다 `fit`을 부르면 핫스팟도 fold마다 새로 고름
- `n_kept_`: 실제로 남은 핫스팟 수를 기록 (결과 표에 표시)

In [3]:
class FrequentTokens(BaseEstimator, TransformerMixin):
    """fold 학습 부분에서 min_count명 미만인 핫스팟 열을 버림. 버릴 목록은 fit에서만 정함."""
    def __init__(self, min_count=10):
        self.min_count = min_count

    def fit(self, X, y=None):
        hs = [c for c in X.columns if c.startswith('hs__')]
        n = X[hs].sum()
        self.drop_ = n.index[n < self.min_count].tolist()
        self.n_kept_ = len(hs) - len(self.drop_)
        return self

    def transform(self, X):
        return X.drop(columns=self.drop_)

def make_hs(min_count):
    return lambda: make_pipeline(FrequentTokens(min_count), BalancedXGB(**BEST))

#### 기준선 vs 핫스팟 추가 (min_count 3가지)
- 첫 줄은 10 최적 설정을 핫스팟 없이 다시 실행 → 0.4113이 재현되는지로 환경 확인
- min_count가 작을수록 드문 변이까지 포함 (열이 많아짐) → 5 / 10 / 20 비교

In [4]:
rows = []
t = time.time()
r = run_cv(lambda: BalancedXGB(**BEST), X_base, y, verbose=False)
rows.append(dict(name='10_best (기준)', n_hs=0, cv=round(r['cv_mean'], 4), std=round(r['cv_std'], 4), sec=round(time.time()-t)))
print(rows[-1])

for mc in [5, 10, 20]:
    t = time.time()
    r = run_cv(make_hs(mc), X_hs, y, verbose=False)
    n_hs = int(np.mean([m.named_steps['frequenttokens'].n_kept_ for m in r['models']]))
    rows.append(dict(name=f'+hotspot_min{mc}', n_hs=n_hs, cv=round(r['cv_mean'], 4),
                     std=round(r['cv_std'], 4), sec=round(time.time()-t)))
    print(rows[-1])

res11 = pd.DataFrame(rows).sort_values('cv', ascending=False)
res11

{'name': '10_best (기준)', 'n_hs': 0, 'cv': 0.4161, 'std': 0.006, 'sec': 142}
{'name': '+hotspot_min5', 'n_hs': 302, 'cv': 0.4209, 'std': 0.0065, 'sec': 158}
{'name': '+hotspot_min10', 'n_hs': 80, 'cv': 0.4213, 'std': 0.0039, 'sec': 147}
{'name': '+hotspot_min20', 'n_hs': 21, 'cv': 0.4178, 'std': 0.0061, 'sec': 154}


,name,n_hs,cv,std,sec
2,+hotspot_min10,80,0.4213,0.0039,147
1,+hotspot_min5,302,0.4209,0.0065,158
3,+hotspot_min20,21,0.4178,0.0061,154
0,10_best (기준),0,0.4161,0.0060,142


#### 어떤 핫스팟이 쓰였는지 (해석용)
- train 전체로 한 번 학습해서 중요도 상위 핫스팟 확인
- 점수 계산용이 아니라, 모델이 의학적으로 알려진 변이를 활용하는지 확인하기 위함

In [5]:
# train 전체로 한 번 학습해서 중요도 상위 핫스팟 확인 (점수 계산용 아님, 해석용)
MC = 10   # res11에서 가장 좋은 min_count로 바꾸기
pipe = make_hs(MC)().fit(X_hs, y)
imp = pd.Series(pipe[-1].feature_importances_, index=pipe[-1].get_booster().feature_names)
print(imp[imp.index.str.startswith('hs__')].sort_values(ascending=False).head(15).round(4))

hs__MEFV_A165A        0.0139
hs__NEFH_P655P        0.0103
hs__LRIG1_L26V        0.0088
hs__LRIG1_L24V        0.0076
hs__ERCC2_D312N       0.0058
hs__PLEC_L1321L       0.0051
hs__CDKN2A_R80stop    0.0044
hs__PLEC_A2113A       0.0041
hs__FGFR3_S249C       0.0037
hs__CDKN2A_R58stop    0.0033
hs__CDKN2A_H83Y       0.0031
hs__SOWAHC_L42L       0.0031
hs__IDH1_R132H        0.0031
hs__TM7SF2_P52P       0.0030
hs__PIK3CA_H1047R     0.0028
dtype: float32


#### 제출 파일 생성
- 셀 5 결과로 고른 min_count로 제출
- test에도 같은 Pipeline이 적용됨 → 각 fold 모델은 자기가 학습할 때 고른 핫스팟 열만 사용

In [6]:
MC = 10   # res11 보고 정하기
r = run_cv(make_hs(MC), X_hs, y, X_hs_test)
save_submission(submission, r['test_proba'], le, name=f'xgb_d3_cs0.3_hotspot{MC}', cv_score=r['cv_mean'])

[fold 0] Macro F1 = 0.4238
[fold 1] Macro F1 = 0.4277
[fold 2] Macro F1 = 0.4182
[fold 3] Macro F1 = 0.4196
[fold 4] Macro F1 = 0.4173
CV Macro F1 = 0.4213 ± 0.0039 (OOF 0.4216)
저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1110_xgb_d3_cs0.3_hotspot10_0.421.csv


WindowsPath('C:/Users/USER/Desktop/hackathon/submissions/1110_xgb_d3_cs0.3_hotspot10_0.421.csv')

## 결과

전처리 변경(PR: 변이 표기 정규화 + dedupe 옵션) 후 다시 실행한 결과입니다.
- any + stats: `load_data()` (이소폼 중복 제거 O) / 핫스팟 토큰: `load_data(dedupe=False)` (표기 통일만)

| 설정 | 핫스팟 수 | CV (± std) | 기준 대비 |
| :--- | --: | :-: | :-: |
| 10 best (기준) | 0 | 0.4161 ± 0.0060 | — |
| +hotspot min5 | 302 | 0.4209 ± 0.0065 | +0.005 |
| **+hotspot min10** | **80** | **0.4213 ± 0.0039** | **+0.005** |
| +hotspot min20 | 21 | 0.4178 ± 0.0061 | +0.002 |
| +hotspot min10 (침묵 제외, 11b) | 67 | 0.4180 ± 0.0052 | +0.0019 |

- 기준선이 0.4113 → 0.4161로 오름: stats를 이소폼 중복 제거 후 데이터로 만든 효과 (핫스팟 없이도 +0.005)
- **세 버전 모두 기준 + std(0.4221) 이내** → 정규화 전(+0.008~0.011, std 초과)과 달리 CV 개선이 오차 범위
  - 기준선이 오르면서 핫스팟이 더해주는 몫이 줄어듦

**min10을 선택한 이유**
- CV(0.4213)가 가장 높고 std(0.0039)도 가장 작음 → 점수와 안정성 모두 최선
- 핫스팟 수 302개 → 80개로 단순 → 드문 변이에 과적합할 여지가 적음

### 참고: 정규화 전 결과(어제 실행)

| 설정 | 핫스팟 수 | CV (± std) | 기준 대비 | LB |
| :--- | --: | :-: | :-: | :-: |
| 10 best (기준) | 0 | 0.4113 ± 0.0024 | — | 0.3206 |
| +hotspot min10 | 80 | 0.4207 ± 0.0038 | +0.009 | 0.3166 |
| +hotspot min10 (침묵 제외, 11b) | 67 | 0.4201 ± 0.0078 | +0.009 | — |

- 정규화 전에는 test의 X 표기 변이(`TP53_R213X` 등)가 train 핫스팟 열(`TP53_R213*`)과 매칭되지 않아 0으로 들어갔음
  - 예: APC_R1450* test 보유 0명 → 정규화 후 25명, TP53_R213* 4명 → 25명
- 11b (정규화 전): 침묵 핫스팟을 빼도 CV 동일(−0.0006)했으나, 정규화 후에는 −0.0033으로 차이가 커져 이 해석은 유지되지 않음


## 중요도 해석

train 전체로 한 번 학습한 모델(min_count 10)의 핫스팟 중요도 상위 15개

| 순위 | 핫스팟 | 중요도 | 유형 |
| :-: | :--- | :-: | :--- |
| 1 | MEFV_A165A | 0.0139 | 침묵 |
| 2 | NEFH_P655P | 0.0103 | 침묵 |
| 3 | LRIG1_L26V | 0.0088 | 미스센스 |
| 4 | LRIG1_L24V | 0.0076 | 미스센스 |
| 5 | ERCC2_D312N | 0.0058 | 미스센스 |
| 6 | PLEC_L1321L | 0.0051 | 침묵 |
| 7 | CDKN2A_R80* | 0.0044 | 절단 |
| 8 | PLEC_A2113A | 0.0041 | 침묵 |
| 9 | FGFR3_S249C | 0.0037 | 미스센스 |
| 10 | CDKN2A_R58* | 0.0033 | 절단 |
| 11 | CDKN2A_H83Y | 0.0031 | 미스센스 |
| 12 | SOWAHC_L42L | 0.0031 | 침묵 |
| 13 | IDH1_R132H | 0.0031 | 미스센스 |
| 14 | TM7SF2_P52P | 0.0030 | 침묵 |
| 15 | PIK3CA_H1047R | 0.0028 | 미스센스 |

- **알려진 암 핫스팟**: CDKN2A R80*·R58*·H83Y, FGFR3 S249C(방광암), IDH1 R132H(뇌종양), PIK3CA H1047R
  - CDKN2A가 3개(절단 2, 미스센스 1)로 가장 많음 → 같은 유전자 안에서 변이 종류가 갈릴 때 핫스팟 피처가 효과가 있음
  - FGFR3 S249C는 정규화 전 4위 → 9위로 내려갔지만 상위 10개 안에 유지
  - BLCA는 05에서 F1 0.188로 약한 아형이었음 → FGFR3 S249C가 도움이 됐을 가능성
- **ERCC2 D312N은 암 핫스팟이 아니라 흔한 생식세포 다형성**(rs1799793)으로 알려진 변이
  - 일반 인구에서도 흔해서 특정 암종을 가리키는 신호로 해석하기 어려움
- **LRIG1 L24V와 L26V는 같은 변이를 이소폼별 번호로 적은 것일 가능성**
  - 핫스팟 토큰은 `dedupe=False` 데이터로 만들었고, `data.py`의 dedupe 설명에서도 LRIG1_L24V가 중복 제거 시 사라지는 후보로 나옴
  - 같은 환자가 두 열을 동시에 갖는지 확인 필요 → 동시 보유율이 높으면 사실상 같은 피처가 2개
- **BRAF V600E가 없는 이유**: BRAF 변이는 대부분 V600E라 기존 유전자 열(BRAF 0/1)이 이미 같은 정보를 가짐
  → 핫스팟 피처는 같은 유전자 안에서 변이 종류가 갈릴 때 효과가 있음
- **주의: 상위 15개 중 6개가 침묵 변이** (MEFV,

## 결론
- **H11 판정 보류**: 정규화 후 CV 개선은 +0.005로 오차 범위 → LB 결과로 판단
- 정규화 전 LB −0.004는 test 핫스팟 열이 일부 비어 있던 상태의 결과라, 핫스팟 자체의 효과로 보기 어려움
- 교훈 (재확인): 피처 추가로 오른 CV는 LB로 안 넘어가는 경향(07 E, 11 정규화 전), 과적합을 줄여서 오른 CV는 넘어감(09, 10)
- 최고 LB는 10 (0.3206) 유지

# 11b - 침묵 변이 빼보기

In [7]:
# 11b. 핫스팟에서 침묵 변이 제외 (any는 그대로 침묵 포함)
from src.features import mutation_type

def patient_tokens_nonsilent(df, genes):
    out = []
    for row in df[genes].to_numpy():
        s = set()
        for g, v in zip(genes, row):
            if v != WT:
                s.update(f'{g}_{t}' for t in v.split() if mutation_type(t) != 'silent')
        out.append(s)
    return out

tok_tr_ns, tok_te_ns = patient_tokens_nonsilent(train_tok, genes), patient_tokens_nonsilent(test_tok, genes)
cnt_ns = Counter(t for s in tok_tr_ns for t in s)
vocab = sorted(t for t, n in cnt_ns.items() if n >= MIN_UNIVERSE)   # token_matrix가 전역 vocab을 쓰므로 교체
print(f'침묵 제외 후보: {len(vocab):,}개')

H_tr_ns, H_te_ns = token_matrix(tok_tr_ns, train.index), token_matrix(tok_te_ns, test.index)
X_hs_ns,  X_hs_ns_test = pd.concat([X_base, H_tr_ns], axis=1), pd.concat([X_base_test, H_te_ns], axis=1)

t = time.time()
r = run_cv(make_hs(10), X_hs_ns, y, verbose=False)
n_hs = int(np.mean([m.named_steps['frequenttokens'].n_kept_ for m in r['models']]))
print(dict(name='+hotspot_min10_nonsilent', n_hs=n_hs, cv=round(r['cv_mean'], 4),
           std=round(r['cv_std'], 4), sec=round(time.time() - t)))

침묵 제외 후보: 369개
{'name': '+hotspot_min10_nonsilent', 'n_hs': 67, 'cv': 0.418, 'std': 0.0052, 'sec': 152}
